# Heavy American Option Pricing with Longstaff-Schwartz (LSM) and parfun

This notebook replaces the simple binomial tree iwth a **Longstaff-Schwartz Monte Carlo (LSM)** mode. LSM introduces:
- A full **stochastic process simulation** (GBM paths)
- **Cross-sectional regressions** at every exercise data
- Much higher computational cost

This is representative of **production American option engines** used for long-dated and complex payoffs.


## Program Structure

![Program Structure](Simple_Deep_American_Option_Parfun_flowchart.svg)

# Native Python Implementation

In [1]:
import os
import sys
import time

import numpy as np

sys.stderr = open(os.devnull, "w")

sequential_runtime = None

PATHS = 80_000
STEPS = 252
DEGREE = 6


def american_put_lsm(S, K, r, sigma, T, steps, paths, degree=6, seed=None):
    if seed is not None:
        np.random.seed(seed)

    dt = T / steps
    disc = np.exp(-r * dt)

    Z = np.random.normal(size=(paths, steps))
    S_paths = np.empty((paths, steps + 1))
    S_paths[:, 0] = S

    for t in range(steps):
        S_paths[:, t + 1] = S_paths[:, t] * np.exp(
            (r - 0.5 * sigma **2) * dt + sigma * np.sqrt(dt) * Z[:, t]
        )

    cashflows = np.maximum(K - S_paths[:, -1], 0.0)

    for t in range(steps - 1, 0, -1):
        itm = S_paths[:, t] < K
        X = S_paths[itm, t]
        Y = cashflows[itm] * disc

        if len(X) > degree:
            coeffs = np.polyfit(X, Y, degree)
            continuation = np.polyval(coeffs, X)
        else:
            continuation = np.zeros_like(X)

        exercise = K - X
        exercise_now = exercise > continuation

        cashflows[itm] = np.where(
            exercise_now,
            exercise,
            Y
        )

    return cashflows.mean() * disc


def price_with_scenarios(p, scenarios):
    S, K, r, _, T, St, N = p
    acc = [american_put_lsm(S, K, r, vol, T, St, N, DEGREE) for vol in scenarios]
    return sum(acc)


def batch_price_with_scenarios(tasks, scenarios):
    return [price_with_scenarios(p, scenarios) / len(scenarios) for p in tasks]


def main():
    global sequential_runtime

    BATCH_SIZE = 4
    SCENARIOS = np.linspace(0.15, 0.35, 20)

    base_param = (100.0, 100.0, 0.05, 0.2, 1.0, STEPS, PATHS)
    BATCH = [base_param] * BATCH_SIZE

    start = time.time()
    results = batch_price_with_scenarios(BATCH, SCENARIOS)
    sequential_runtime = time.time() - start

    print(f"Sequential: {sequential_runtime / 60:.2f} minutes")


if __name__ == "__main__":
    main()

Sequential: 6.08 minutes


/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_835774/1492503087.py:42: Ran

# Parfun Implementation

In [2]:
import os
import sys
import time

import numpy as np

import parfun as pf

sys.stderr = open(os.devnull, "w")

parallel_runtime = None

PATHS = 80_000
STEPS = 252
DEGREE = 6


def american_put_lsm(S, K, r, sigma, T, steps, paths, degree=6, seed=None):
    if seed is not None:
        np.random.seed(seed)

    dt = T / steps
    disc = np.exp(-r * dt)

    Z = np.random.normal(size=(paths, steps))
    S_paths = np.empty((paths, steps + 1))
    S_paths[:, 0] = S

    for t in range(steps):
        S_paths[:, t + 1] = S_paths[:, t] * np.exp(
            (r - 0.5 * sigma **2) * dt + sigma * np.sqrt(dt) * Z[:, t]
        )

    cashflows = np.maximum(K - S_paths[:, -1], 0.0)

    for t in range(steps - 1, 0, -1):
        itm = S_paths[:, t] < K
        X = S_paths[itm, t]
        Y = cashflows[itm] * disc

        if len(X) > degree:
            coeffs = np.polyfit(X, Y, degree)
            continuation = np.polyval(coeffs, X)
        else:
            continuation = np.zeros_like(X)

        exercise = K - X
        exercise_now = exercise > continuation

        cashflows[itm] = np.where(
            exercise_now,
            exercise,
            Y
        )

    return cashflows.mean() * disc


def price_with_scenarios(p, scenarios):
    S, K, r, _, T, St, N = p
    acc = [american_put_lsm(S, K, r, vol, T, St, N, DEGREE) for vol in scenarios]
    return sum(acc)


@pf.parfun(
    split=pf.per_argument(tasks=pf.py_list.by_chunk),
    combine_with=pf.py_list.concat,
    fixed_partition_size=1,
)
def batch_price_with_scenarios(tasks, scenarios):
    return [price_with_scenarios(p, scenarios) / len(scenarios) for p in tasks]


def main():
    global parallel_runtime

    BATCH_SIZE = 4
    SCENARIOS = np.linspace(0.15, 0.35, 20)

    base_param = (100.0, 100.0, 0.05, 0.2, 1.0, STEPS, PATHS)
    BATCH = [base_param] * BATCH_SIZE

    with pf.set_parallel_backend_context("scaler_local", n_workers=4):
        start = time.time()
        results = batch_price_with_scenarios(BATCH, SCENARIOS)
        parallel_runtime = time.time() - start

    print(f"Parallel:   {parallel_runtime / 60:.2f} minutes")


if __name__ == "__main__":
    main()

[INFO]2026-03-31 10:49:55+0800: logging to ('/dev/stdout',)
[INFO]2026-03-31 10:49:55+0800: ObjectStorageServer: start and listen to tcp://127.0.0.1:60793
[INFO]2026-03-31 10:49:55+0800: ObjectStorageServer: started
[INFO]2026-03-31 10:49:55+0800: logging to ('/dev/stdout',)
[INFO]2026-03-31 10:49:55+0800: use event loop: builtin
[INFO]2026-03-31 10:49:55+0800: ConfigController: scheduler_address = tcp://127.0.0.1:35457
[INFO]2026-03-31 10:49:55+0800: ConfigController: object_storage_address = tcp://127.0.0.1:60793
[INFO]2026-03-31 10:49:55+0800: ConfigController: monitor_address = None
[INFO]2026-03-31 10:49:55+0800: ConfigController: protected = True
[INFO]2026-03-31 10:49:55+0800: ConfigController: max_number_of_tasks_waiting = -1
[INFO]2026-03-31 10:49:55+0800: ConfigController: client_timeout_seconds = 60
[INFO]2026-03-31 10:49:55+0800: ConfigController: worker_timeout_seconds = 60
[INFO]2026-03-31 10:49:55+0800: ConfigController: object_retention_seconds = 60
[INFO]2026-03-31 10:

# Diff: Native vs Parfun

In [3]:
import re
import difflib
from IPython.display import display, HTML

native_code = In[1]
parfun_code = In[2]

differ = difflib.HtmlDiff(wrapcolumn=90)
table = differ.make_table(
    native_code.splitlines(),
    parfun_code.splitlines(),
    fromdesc="Native Python",
    todesc="With Parfun",
    context=False,
)

# Strip unwanted columns: navigation links, line numbers, and nowrap attribute
table = re.sub(r'<td class="diff_next"[^>]*>.*?</td>', '', table)
table = re.sub(r'<th class="diff_next"[^>]*>.*?</th>', '', table)
table = re.sub(r'<td class="diff_header"[^>]*>.*?</td>', '', table)
table = table.replace('colspan="2"', '')
table = table.replace(' nowrap="nowrap"', '')
table = re.sub(r'(\s*<colgroup></colgroup>){6}',
               '\n        <colgroup></colgroup> <colgroup></colgroup>', table)

css = """<style>
    table.diff { font-family: monospace; font-size: 10px; border-collapse: collapse; width: 100%; }
    table.diff td { padding: 2px 8px; white-space: pre-wrap; word-break: break-all; text-align: left !important; }
    table.diff th { padding: 6px 8px; text-align: center; background-color: #f0f0f0; font-size: 14px; }
    td.diff_add { background-color: #e6ffec; }
    td.diff_chg { background-color: #fff8c5; }
    td.diff_sub { background-color: #ffebe9; }
    span.diff_add { background-color: #ccffd8; }
    span.diff_chg { background-color: #fff2a8; }
    span.diff_sub { background-color: #ffc0c0; }
</style>"""

display(HTML(css + table))

Native Python,With Parfun
import os,import os
import sys,import sys
import time,import time
,
import numpy as np,import numpy as np
,
,import parfun as pf
,
"sys.stderr = open(os.devnull, ""w"")","sys.stderr = open(os.devnull, ""w"")"
,


# Speedup

In [4]:
print(f"Sequential: {sequential_runtime / 60:.2f} minutes")
print(f"Parallel:   {parallel_runtime / 60:.2f} minutes")
print(f"Speedup:    {sequential_runtime / parallel_runtime:.2f}x")

Sequential: 6.08 minutes
Parallel:   4.88 minutes
Speedup:    1.24x
